# 2-3절 연습 문제 풀이

이 노트북은 2-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch02/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

## 연습 2-7

1장에서 외계 행성의 물리 법칙을 검증하려고 직접 구현한 회귀 분석 모델을, 파이토치 고수준 API로 다시 만들어 보자.

In [ ]:
# 1장의 회귀 분석 모델을 고수준 API로 다시 만든다.
g = torch.Generator().manual_seed(SEED)
x = torch.rand(50, 1, generator=g) * 10
y_true = 4.9 * x ** 2
y = y_true + torch.randn(50, 1, generator=g) * (y_true * 0.1)

# y = a*x^2 + b 는 x^2을 입력으로 하는 선형 계층 하나로 표현된다.
model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-4)

x_squared = x ** 2
for epoch in range(2000):
    loss = criterion(model(x_squared), y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

a = model.weight.item(); b = model.bias.item()
print(f'학습된 a: {a:.4f} (실제 4.9)')
print(f'학습된 b: {b:.4f} (실제 0)')

`nn.Linear(1, 1)`의 가중치가 a, 편향이 b에 대응한다. 입력으로 x가 아니라 **x²**을 넣는 것이 핵심이다. 직접 구현하던 순전파, 손실 계산, 파라미터 갱신이 각각 `model()`, `nn.MSELoss()`, `optimizer.step()`으로 대체된다.

## 연습 2-8

입력이 세 개인 [연습 문제 2-6]의 AND 게이트 시뮬레이터 모델을 파이토치 고수준 API로 다시 만들어 보자.

In [ ]:
# 입력이 세 개인 AND 게이트 - 고수준 API 버전
X3 = torch.tensor([[float(b) for b in f'{i:03b}'] for i in range(8)])
Y3 = (X3.sum(dim=1, keepdim=True) == 3).float()

model = nn.Sequential(nn.Linear(3, 1), nn.Sigmoid())
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(3000):
    loss = criterion(model(X3), Y3)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

pred = (model(X3) > 0.5).float()
print(f'정답 여부: {torch.equal(pred, Y3)}')
print(f'가중치: {[round(v, 2) for v in model[0].weight.flatten().tolist()]}, '
      f'편향: {model[0].bias.item():.2f}')

세 가중치가 모두 비슷한 양수, 편향은 큰 음수로 학습된다. 세 입력이 모두 1이어야 가중합이 편향을 넘어 양수가 되는 구조다.

## 연습 2-9

0과 1의 조합을 입력받아, 한 출력은 AND 게이트를, 다른 출력은 OR 게이트를 시뮬레이션하는 모델을 만들어 보자. 이 모델은 입력 두 개와 출력 두 개를 가진다.

In [ ]:
# 출력이 두 개: 하나는 AND, 하나는 OR
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0., 0.],      # AND, OR
                  [0., 1.],
                  [0., 1.],
                  [1., 1.]])

model = nn.Sequential(nn.Linear(2, 2), nn.Sigmoid())   # 출력 크기만 2로 지정
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(5000):
    loss = criterion(model(X), Y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

pred = (model(X) > 0.5).float()
print(f'{"입력":>8} {"AND":>5} {"OR":>4}')
for x, p in zip(X.tolist(), pred.tolist()):
    print(f'{str([int(v) for v in x]):>8} {int(p[0]):>5} {int(p[1]):>4}')
print(f'\n전체 정답 여부: {torch.equal(pred, Y)}')

출력 뉴런 두 개가 **같은 입력을 공유하면서 서로 다른 결정 경계**를 학습한다. 두 게이트가 각각 선형 분리 가능하므로 한 모델로 동시에 학습할 수 있다. 이렇게 출력 크기를 늘리는 방식이 3장 이후 다중 클래스 분류로 이어진다.

## 연습 2-10

XOR 게이트는 두 입력이 같으면 0을, 다르면 1을 출력한다. 이 XOR 게이트를 시뮬레이션하는 퍼셉트론 모델을 만들어 보자.

만약 학습된 퍼셉트론의 분류 결과가 좋지 않다면, 그 이유가 무엇일지 추론해 제시해 보자. 참고로 퍼셉트론 모델로 이 데이터를 제대로 분류할 수 없으며, 이를 극복하는 것이 다음 3장의 주요 목표 중 하나다.

2장 학습 노트

In [ ]:
# XOR - 두 입력이 다를 때만 1
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y_xor = torch.tensor([[0.], [1.], [1.], [0.]])

model = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
for epoch in range(10000):
    loss = criterion(model(X), Y_xor)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

pred = model(X)
print(f'최종 손실: {loss.item():.4f}')
for x, p, y in zip(X.tolist(), pred.flatten().tolist(), Y_xor.flatten().tolist()):
    print(f'입력 {[int(v) for v in x]} -> 출력 {p:.3f} (정답 {int(y)})')

# 결정 경계를 그려 확인한다.
w = model[0].weight.detach().flatten(); b = model[0].bias.detach()
print(f'\n결정 경계: {w[0]:.2f}*x1 + {w[1]:.2f}*x2 + {b.item():.2f} = 0')

네 출력이 모두 0.5 부근에 머물러 어느 쪽으로도 분류하지 못한다.

**이유**: 퍼셉트론의 결정 경계는 **직선 하나**뿐인데, XOR의 정답 (0,1)·(1,0)과 (0,0)·(1,1)은 서로 대각선 위치에 놓여 있다. 어떤 직선을 그어도 두 집합을 갈라놓을 수 없다(선형 분리 불가능).

이 한계가 1차 인공지능 겨울의 계기가 되었고, 이를 넘어서는 방법이 3장의 **다층 퍼셉트론**이다. 은닉층을 하나 두면 결정 경계를 여러 개 만들어 조합할 수 있다.